In [ ]:
import warnings
warnings.filterwarnings("ignore")

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd
import numpy as np

print("Ambiente pronto. Bibliotecas importadas.")

In [ ]:
# =============================================================================
# Carregamento dos dados do dataProcessing
# =============================================================================
# Assume que corriste dataProcessing e salvaste com to_pickle
# Ajusta caminhos/nomes se necessário

base_dir = Path("../data/processed")
df_profiles   = pd.read_pickle(base_dir / "processed_profiles.pkl")
df_influencers = pd.read_pickle(base_dir / "processed_influencers.pkl")

print(f"Perfis carregados: {df_profiles.shape}")
print(f"Posts carregados:  {df_influencers.shape}")

In [ ]:
# =============================================================================
# 1. Embeddings Textuais (Semântica)
# =============================================================================
# Modelo: all-MiniLM-L6-v2 → leve (384 dims), multilíngue, eficiente para LinkedIn
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Modelo SentenceTransformer carregado.")

def get_text_embedding(text_series, prefix=""):
    """Gera embeddings e retorna DataFrame alinhado"""
    text_series = text_series.fillna("No description available").replace("", "No description available")
    embeddings = model.encode(text_series.tolist(), show_progress_bar=True, batch_size=32)
    df_emb = pd.DataFrame(embeddings, index=text_series.index)
    df_emb.columns = [f"emb_{prefix}_{i}" for i in range(df_emb.shape[1])]
    return df_emb

# Perfis: headline + about
df_profiles['profile_text'] = df_profiles['headline'].fillna("") + " " + df_profiles['about'].fillna("")
profile_embeddings = get_text_embedding(df_profiles['profile_text'], "prof")

# Posts: conteúdo principal
post_embeddings = get_text_embedding(df_influencers['content'], "post")

print(f"Embeddings → Perfis: {profile_embeddings.shape}, Posts: {post_embeddings.shape}")

In [ ]:
# =============================================================================
# 4. Pré-processadores Tabulares
# =============================================================================
num_p = ['num_skills', 'num_experience', 'num_education', 'about_length', 'headline_length', 'connections', 'years_experience']
cat_p = ['seniority_level', 'industry']

preprocessor_p = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_p),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_p)
    ])

num_po = ['followers', 'num_hashtags', 'reactions', 'comments', 'time_spent']
cat_po = ['media_type', 'location']

preprocessor_po = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_po),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_po)
    ])

In [ ]:
# =============================================================================
# 5. Features Finais (Tabular + PCA)
# =============================================================================
X_profiles = pd.DataFrame(preprocessor_p.fit_transform(df_profiles), index=df_profiles.index)
y_profiles = df_profiles['profile_discoverability_score']

X_posts = pd.DataFrame(preprocessor_po.fit_transform(df_influencers), index=df_influencers.index)
y_posts = df_influencers['post_discoverability_score']

print(f"Final: Perfis {X_profiles.shape}, Posts {X_posts.shape}")

In [ ]:
# =============================================================================
# 6. Splits Estratificados
# =============================================================================
def safe_split(X, y):
    y_bins = pd.qcut(y, q=5, labels=False, duplicates='drop')
    X_tr, X_temp, y_tr, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y_bins)
    X_val, X_te, y_val, y_te = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_bins[y_temp.index])
    return X_tr, X_val, X_te, y_tr, y_val, y_te

X_train_p, X_val_p, X_test_p, y_train_p, y_val_p, y_test_p = safe_split(X_profiles, y_profiles)
X_train_po, X_val_po, X_test_po, y_train_po, y_val_po, y_test_po = safe_split(X_posts, y_posts)

print(f"Perfis: Train {X_train_p.shape}, Val {X_val_p.shape}, Test {X_test_p.shape}")
print(f"Posts:  Train {X_train_po.shape}, Val {X_val_po.shape}, Test {X_test_po.shape}")

In [ ]:
# =============================================================================
# 7. EXPORT FINAL – Treino / Validação / Teste (formato original)
# =============================================================================
base_dir = Path("../data")
train_dir = base_dir / "train"
val_dir   = base_dir / "validation"
test_dir  = base_dir / "test"

for d in [train_dir, val_dir, test_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Perfis
X_train_p_df = pd.DataFrame(X_train_p).astype(np.float32)
y_train_p_df = pd.DataFrame(y_train_p, columns=['profile_discoverability_score']).astype(np.float32)
X_val_p_df   = pd.DataFrame(X_val_p).astype(np.float32)
y_val_p_df   = pd.DataFrame(y_val_p, columns=['profile_discoverability_score']).astype(np.float32)
X_test_p_df  = pd.DataFrame(X_test_p).astype(np.float32)
y_test_p_df  = pd.DataFrame(y_test_p, columns=['profile_discoverability_score']).astype(np.float32)

X_train_p_df.to_csv(train_dir / "X_profiles.csv", index=False)
y_train_p_df.to_csv(train_dir / "y_profiles.csv", index=False)
X_val_p_df.to_csv(val_dir / "X_profiles.csv", index=False)
y_val_p_df.to_csv(val_dir / "y_profiles.csv", index=False)
X_test_p_df.to_csv(test_dir / "X_profiles.csv", index=False)
y_test_p_df.to_csv(test_dir / "y_profiles.csv", index=False)

# Posts
X_train_po_df = pd.DataFrame(X_train_po).astype(np.float32)
y_train_po_df = pd.DataFrame(y_train_po, columns=['post_discoverability_score']).astype(np.float32)
X_val_po_df   = pd.DataFrame(X_val_po).astype(np.float32)
y_val_po_df   = pd.DataFrame(y_val_po, columns=['post_discoverability_score']).astype(np.float32)
X_test_po_df  = pd.DataFrame(X_test_po).astype(np.float32)
y_test_po_df  = pd.DataFrame(y_test_po, columns=['post_discoverability_score']).astype(np.float32)

X_train_po_df.to_csv(train_dir / "X_posts.csv", index=False)
y_train_po_df.to_csv(train_dir / "y_posts.csv", index=False)
X_val_po_df.to_csv(val_dir / "X_posts.csv", index=False)
y_val_po_df.to_csv(val_dir / "y_posts.csv", index=False)
X_test_po_df.to_csv(test_dir / "X_posts.csv", index=False)
y_test_po_df.to_csv(test_dir / "y_posts.csv", index=False)

print("Export concluído!")
print("Estrutura:")
print("  ../data/train/X_profiles.csv   y_profiles.csv")
print("  ../data/validation/X_profiles.csv   y_profiles.csv")
print("  ../data/test/X_profiles.csv   y_profiles.csv")
print("  (o mesmo para _posts.csv)")

In [ ]:
# =============================================================================
# Resumo para Relatório / Apresentação Oral
# =============================================================================
print("\n=== Resumo Final ===")
print("Datasets prontos para modelTraining:")
print(f"Perfis: Train {X_train_p.shape}, Val {X_val_p.shape}, Test {X_test_p.shape}")
print(f"Posts:  Train {X_train_po.shape}, Val {X_val_po.shape}, Test {X_test_po.shape}")
print("Features: tabular (num/cat) + PCA embeddings (semântica)")
print("Export em CSV float32, pastas separadas.")
print("Alinhamento: Data Preparation completa (ETL, splits estratificados, export).")